# 03 - Export reviewed model artifacts

**SCAFFOLD - NOT RUN.** Export only a model selected on reviewed held-out data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q joblib skl2onnx onnxruntime

import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
from pathlib import Path
import hashlib, json, joblib

ARTIFACT_DIR = Path('/content/drive/MyDrive/terrasafe/export')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
# Set these from the approved training/evaluation notebook; do not export synthetic-only models as operational.
model = selected_model
feature_columns = reviewed_feature_columns
data_source = reviewed_dataset_identifier
joblib_path = ARTIFACT_DIR / 'susceptibility.joblib'
joblib.dump({'model': model, 'feature_columns': feature_columns, 'model_name': type(model).__name__, 'synthetic': False, 'data_source': data_source}, joblib_path)

# ONNX conversion is model-specific; run only after checking converter support and parity on fixed fixtures.
# Compute a checksum manifest after artifact validation.
digest = hashlib.sha256(joblib_path.read_bytes()).hexdigest()
manifest = {'models': [{'name': joblib_path.name, 'url': 'REPLACE_WITH_TRUSTED_HTTPS_RELEASE_URL', 'sha256': digest}]}
(ARTIFACT_DIR / 'model_manifest.json').write_text(json.dumps(manifest, indent=2))

In [ ]:
# Optional compact ONNX export; model converter support and parity must be verified.
try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    onnx_model = convert_sklearn(
        model,
        initial_types=[('features', FloatTensorType([None, len(feature_columns)]))],
    )
    (ARTIFACT_DIR / 'susceptibility.onnx').write_bytes(onnx_model.SerializeToString())
except (ImportError, TypeError, ValueError) as error:
    print(f'ONNX export unavailable for this estimator: {error}')

Create and review a model card with dataset/licence, region and dates, features, spatial/temporal split, measured metrics with denominators, false-negative analysis, calibration, model version, and SHA-256. Joblib artifacts are trusted-code-only.

No model export or ONNX conversion has been run.